In [ ]:
import sys
import os
import matplotlib.pyplot as plt
import numpy as np

sys.path.append(os.path.abspath('../code/methods/datasets'))
sys.path.append(os.path.abspath('../code/methods/losses'))
sys.path.append(os.path.abspath('../code'))

from DatasetBlosc2 import DatasetBlosc2
from losses import ssim_loss

In [ ]:
dataset_path = 'D:/data/BraTS2023-MEN-adapted-Blosc2/train'

dataset = DatasetBlosc2(folder=dataset_path,
                        identifiers=None)

In [ ]:
idx = dataset.identifiers
data, seg = dataset.load_case(idx[0])

In [ ]:
# plot 
fig, ax = plt.subplots(1, 4, figsize=(15, 5))
ax[0].imshow(data[0, :, :, 70], cmap='gray')
ax[0].set_title('t1c')
ax[1].imshow(data[1, :, :, 70], cmap='gray')
ax[1].set_title('t1n')
ax[2].imshow(data[2, :, :, 70], cmap='gray')
ax[2].set_title('t2w')
ax[3].imshow(data[3, :, :, 70], cmap='gray')
ax[3].set_title('t2f')
plt.show()

In [ ]:
from torchvision import transforms
import torchio as tio
import torch


# apply transform
transform = transforms.Compose([
    transforms.Lambda(lambda x: x.unsqueeze(0)),
    transforms.Lambda(lambda x: tio.RandomFlip(axes=1, p=0.5)(x)),  # Random horizontal flip
    transforms.Lambda(lambda x: tio.RandomAffine(scales=(0.8, 1.2), degrees=(-15, 15), translation=0, center='image', image_interpolation='linear', p=0.5)(x)),  # Random affine transformation
    transforms.Lambda(lambda x: tio.RandomGamma(log_gamma=(0.5), p=0.5)(x)),  # Random gamma adjustment
    transforms.Lambda(lambda x: (x - x.min()) / (x.max() - x.min()) * 2 - 1),  # Normalize to [-1, 1]
])

transformed_data = transform(torch.tensor(data[0, :, :], dtype=torch.float32))

print("Transformed data shape:", transformed_data.shape)
print("Transformed data min:", transformed_data.min())
print("Transformed data max:", transformed_data.max())


In [ ]:
# Plot histograms of original and transformed data
fig, ax = plt.subplots(figsize=(10, 5))

# Flatten the data for histogram
original_data_flat = data[0, :, :, 70].flatten()
transformed_data_flat = transformed_data[0, :, :, 70].numpy().flatten()

# Plot histograms
ax.hist(original_data_flat[original_data_flat>0], bins=50, alpha=0.5, label='Original t1c', color='blue')
ax.hist(transformed_data_flat[transformed_data_flat>-1], bins=50, alpha=0.5, label='Transformed t1c', color='orange')

# Add legend and labels
ax.set_title('Histogram of Original and Transformed t1c')
ax.set_xlabel('Intensity')
ax.set_ylabel('Frequency')
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# plot transformed data
fig, ax = plt.subplots(2, 1, figsize=(15, 10))
# Original data
ax[0].imshow(data[0, :, :, 70], cmap='gray')
ax[0].set_title('Original t1c')

# Transformed data
ax[1].imshow(transformed_data[0, :, :,70].numpy(), cmap='gray')
ax[1].set_title('Transformed t1c')

plt.tight_layout()
plt.show()